In [ ]:
# In[1]:

import json

# Load the task.json file to understand the alert context
task_directory = 'rca100/cases/t087'
task_file_path = f'{task_directory}/task.json'

with open(task_file_path, 'r') as file:
    task_data = json.load(file)

alert_context = task_data
alert_context

```
Out[1]:
```
The alert is titled "checkout 错误次数告警" and was triggered on April 28, 2026, at 18:27:26 (UTC+8). The alert time window spans from 18:25:17 to 18:34:26 (UTC+8). The affected entity is the operation `checkout::/oteldemo.CheckoutService/PlaceOrder` of type `apm.operation` in the APM domain. The user requests an analysis of the root cause.

The original code execution output of IPython Kernel is also provided below for reference:

{'task_id': 't087', 'task_version': '1.0', 'alert_event_id': 'o5uu0m91594kn5slhis6pcm0i6', 'alert_title': 'checkout 错误次数告警', 'alert_trigger_time': '2026-04-28T18:27:26+0800', 'alert_window': {'start': '2026-04-28T18:25:17.392334+08:00', 'end': '2026-04-28T18:34:26+08:00'}, 'alert_entity': {'entity_id': 'd219413245b68b297976412bbee076cf', 'entity_name': 'checkout::/oteldemo.CheckoutService/PlaceOrder', 'entity_type': 'apm.operation', 'entity_domain': 'apm'}, 'prompt_text': '<code vibeops_object type="alert_event">checkout 错误次数告警<alert_event event_id="413a01688090b0fe965068130e7e000c" trans_id="o5uu0m91594kn5slhis6pcm0i6" rule_id="08493be2-a181-4480-9895-2b2c5607122e" rule_name="checkout 错误次数告警" entity_id="d219413245b68b297976412bbee076cf" entity_name="checkout::/oteldemo.CheckoutService/PlaceOrder" entity_type="apm.operation" entity_domain="apm" alert_time="2026-04-28T18:27:26+08:00" current_value="6523.0" /></code> 帮我分析下根因。', 'workspace': 'rca-benchmark', 'region_id': 'cn-hongkong', 'available_modalities': ['metrics', 'logs', 'traces', 'events', 'alerts', 'topology'], 'scoring_note': 'Output contract (prediction_schema.json) and fault taxonomy (taxonomy.json) will be published in a follow-up release.'}```
```

In [ ]:
# In[2]:

The Executor failed to complete the instruction, please re-write a new instruction for Executor.

```
Out[2]:
```
The Executor failed to complete the instruction, please re-write a new instruction for Executor.```
```

In [ ]:
# In[3]:

The Executor failed to complete the instruction, please re-write a new instruction for Executor.

```
Out[3]:
```
The Executor failed to complete the instruction, please re-write a new instruction for Executor.```
```

In [ ]:
# In[4]:

import pandas as pd

# Load the metrics.parquet file
metrics_file_path = f'{task_directory}/metrics.parquet'
metrics_data = pd.read_parquet(metrics_file_path)

# Ensure the 'time' and 'value' columns are numeric for proper processing
metrics_data['time'] = pd.to_numeric(metrics_data['time'], errors='coerce')
metrics_data['value'] = pd.to_numeric(metrics_data['value'], errors='coerce')

# Define the KPIs to calculate global thresholds for
kpi_list = ['request_count', 'error_count', 'error_rate', 'latency', 'slow', 'slow_count', 
            'node_cpu_usage_rate', 'node_memory_usage_rate', 'node_disk_usage_rate', 
            'cpu', 'mem', 'workload', 'deployment_desired_replicas', 
            'deployment_ready_replicas', 'deployment_available_replicas', 
            'deployment_availability_rate', 'deployment_cpu_usage_total', 
            'deployment_memory_usage_total']

# Filter metrics for the relevant KPIs
filtered_metrics = metrics_data[metrics_data['metric'].isin(kpi_list)]

# Calculate global thresholds (P95) for each KPI
global_thresholds = filtered_metrics.groupby('metric')['value'].quantile(0.95).reset_index()
global_thresholds.rename(columns={'value': 'p95_threshold'}, inplace=True)

global_thresholds

```
Out[4]:
```
The global P95 thresholds for key performance indicators (KPIs) have been calculated. Notable thresholds include:

- `error_rate`: 100% (1.0)
- `error_count`: 18,929
- `latency`: 0.0345 seconds
- `request_count`: 38,455
- `node_cpu_usage_rate`: 21.78%
- `node_memory_usage_rate`: 88.53%
- `deployment_cpu_usage_total`: 1.28 cores
- `deployment_memory_usage_total`: 1.57 GB

These thresholds will serve as benchmarks to identify anomalies in the metrics during further analysis.

The original code execution output of IPython Kernel is also provided below for reference:

metric  p95_threshold
0                             cpu   6.210874e-01
1    deployment_availability_rate   1.000000e+02
2   deployment_available_replicas   3.000000e+00
3      deployment_cpu_usage_total   1.284553e+00
4     deployment_desired_replicas   3.000000e+00
5   deployment_memory_usage_total   1.574476e+09
6       deployment_ready_replicas   3.000000e+00
7                     error_count   1.892900e+04
8                      error_rate   1.000000e+00
9                         latency   3.455386e-02
10                            mem   6.581707e+08
11            node_cpu_usage_rate   2.177736e+01
12           node_disk_usage_rate   3.735352e+01
13         node_memory_usage_rate   8.853259e+01
14                  request_count   3.845580e+04
15                           slow   2.530000e+02
16                     slow_count   3.654000e+02
17                       workload   8.952520e+04```
```

In [ ]:
# In[5]:

import pytz
from datetime import datetime

# Define the alert time window in UTC+8
alert_start = datetime.fromisoformat('2026-04-28T18:25:17.392334+08:00').astimezone(pytz.timezone('Asia/Shanghai'))
alert_end = datetime.fromisoformat('2026-04-28T18:34:26+08:00').astimezone(pytz.timezone('Asia/Shanghai'))

# Convert the alert time window to Unix microseconds for filtering
alert_start_us = int(alert_start.timestamp() * 1_000_000)
alert_end_us = int(alert_end.timestamp() * 1_000_000)

# Filter metrics data to include only the alert time window
filtered_metrics_window = filtered_metrics[
    (filtered_metrics['time'] >= alert_start_us) & (filtered_metrics['time'] <= alert_end_us)
]

# Merge the filtered metrics with global thresholds to identify anomalies
anomalies = filtered_metrics_window.merge(global_thresholds, on='metric')
anomalies['is_anomalous'] = anomalies['value'] > anomalies['p95_threshold']

# Group by entity and metric to detect consecutive anomalies
anomalies_grouped = anomalies[anomalies['is_anomalous']].groupby(['entity_name', 'metric'])

# Count consecutive anomalies for each entity and metric
anomalies_count = anomalies_grouped.size().reset_index(name='anomaly_count')

# Filter for entities and metrics with significant consecutive anomalies
significant_anomalies = anomalies_count[anomalies_count['anomaly_count'] > 1]

significant_anomalies

```
Out[5]:
```
The analysis identified several components with significant consecutive anomalies during the alert time window. Key findings include:

- Deployment-level anomalies: High `deployment_cpu_usage_total` (24 anomalies) and `deployment_memory_usage_total` (36 anomalies).
- Node-level anomalies: Elevated `node_cpu_usage_rate` (2 anomalies), `node_disk_usage_rate` (18 anomalies), and `node_memory_usage_rate` (4 anomalies).
- Service-level anomalies:
  - `checkout::/oteldemo.CheckoutService/PlaceOrder`: High `error_count` (6 anomalies).
  - `frontend`: High `cpu` (18 anomalies) and `workload` (16 anomalies).
  - `frontend-proxy::ingress`: High `error_count` (5 anomalies) and `request_count` (16 anomalies).
  - `email`: High `mem` usage (18 anomalies).
  - `accounting`: High `latency` (11 anomalies).
  - `product-catalog`: High `request_count` (3 anomalies).

These anomalies suggest potential faults in the `checkout`, `frontend`, and `frontend-proxy` services, as well as resource pressure at the deployment and node levels. Further investigation is needed to pinpoint the root cause.

The original code execution output of IPython Kernel is also provided below for reference:

entity_name                         metric  anomaly_count
0                                                         deployment_cpu_usage_total             24
1                                                      deployment_memory_usage_total             36
2                                                                node_cpu_usage_rate              2
3                                                               node_disk_usage_rate             18
4                                                             node_memory_usage_rate              4
5                                          accounting                        latency             11
6      checkout::/oteldemo.CheckoutService/PlaceOrder                    error_count              6
7                                               email                            mem             18
8                                            frontend                            cpu             18
9                                            frontend                       workload             16
10                          frontend-77ff944f66-2prmd                  request_count             16
11                          frontend-77ff944f66-96fld                  request_count             16
12                          frontend-77ff944f66-xcxnx                  request_count             16
13                            frontend-proxy::ingress                    error_count              5
14                            frontend-proxy::ingress                  request_count             16
15                                      frontend::GET                  request_count              3
16                                     frontend::POST                    error_count              5
17                       frontend::POST /api/checkout                    error_count              5
18  product-catalog::oteldemo.ProductCatalogServic...                  request_count              3```
```

In [ ]:
# In[6]:

# Load the traces.parquet file
traces_file_path = f'{task_directory}/traces.parquet'
traces_data = pd.read_parquet(traces_file_path)

# Ensure the 'startTime', 'endTime', and 'duration' columns are numeric for proper processing
traces_data['startTime'] = pd.to_numeric(traces_data['startTime'], errors='coerce')
traces_data['endTime'] = pd.to_numeric(traces_data['endTime'], errors='coerce')
traces_data['duration'] = pd.to_numeric(traces_data['duration'], errors='coerce')

# Convert the alert time window to Unix nanoseconds for filtering
alert_start_ns = alert_start_us * 1_000
alert_end_ns = alert_end_us * 1_000

# Filter traces within the alert time window
filtered_traces = traces_data[
    (traces_data['startTime'] >= alert_start_ns) & (traces_data['endTime'] <= alert_end_ns)
]

# Focus on traces related to the `checkout::/oteldemo.CheckoutService/PlaceOrder` operation
checkout_traces = filtered_traces[
    filtered_traces['spanName'] == '/oteldemo.CheckoutService/PlaceOrder'
]

# Analyze the call chain to determine the most downstream faulty service
# A faulty service can be identified by non-zero statusCode or error messages in statusMessage
checkout_traces_faulty = checkout_traces[
    (checkout_traces['statusCode'] != 0) | (checkout_traces['statusMessage'].notna())
]

# Group by serviceName to identify the most downstream faulty service
faulty_services = checkout_traces_faulty.groupby('serviceName').size().reset_index(name='fault_count')
faulty_services_sorted = faulty_services.sort_values(by='fault_count', ascending=False)

faulty_services_sorted

```
Out[6]:
```
The trace analysis within the alert time window reveals that the `checkout` service is the most downstream faulty service, with 4,395 fault occurrences related to the `/oteldemo.CheckoutService/PlaceOrder` operation. This indicates that the `checkout` service is likely the root cause of the issue.

The original code execution output of IPython Kernel is also provided below for reference:

serviceName  fault_count
0    checkout         4395```
```

In [ ]:
# In[7]:

# Load the logs.parquet file
logs_file_path = f'{task_directory}/logs.parquet'
logs_data = pd.read_parquet(logs_file_path)

# Ensure the '_time_' column is in a consistent format for filtering
logs_data['_time_'] = pd.to_datetime(logs_data['_time_'], errors='coerce')

# Filter logs within the alert time window
filtered_logs = logs_data[
    (logs_data['_time_'] >= alert_start) & (logs_data['_time_'] <= alert_end)
]

# Focus on logs related to the `checkout` service
checkout_logs = filtered_logs[
    filtered_logs['_container_name_'].str.contains('checkout', case=False, na=False)
]

# Look for error messages or anomalies in the logs
checkout_error_logs = checkout_logs[
    checkout_logs['content'].str.contains('error', case=False, na=False)
]

checkout_error_logs

```
Out[7]:
```
No error logs were found for the `checkout` service within the alert time window. This suggests that the root cause may not be directly observable in the logs or could be related to other telemetry data such as metrics or traces. Further investigation is needed to confirm the root cause.

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: [content, _time_, _source_, _container_ip_, _image_name_, _container_name_, _pod_name_, _namespace_, _pod_uid_, __topic__, __tag__:__pack_id__, __tag__:_node_name_, __tag__:_node_ip_, __tag__:_cluster_id_, __tag__:__hostname__]
Index: []

[0 rows x 15 columns]```
```

In [ ]:
# In[8]:

# Reload the events.parquet file
events_data = pd.read_parquet(events_file_path)

# Inspect the columns to identify the correct timestamp column
events_columns = events_data.columns

events_columns

```
Out[8]:
```
The `events.parquet` file contains columns such as `eventId`, `hostname`, `level`, `pod_id`, `pod_name`, `clusterId`, and `clusterName`. However, it does not include a direct timestamp column for filtering by time. Further investigation is needed to determine how to filter events within the alert time window.

The original code execution output of IPython Kernel is also provided below for reference:

Index(['eventId', 'hostname', 'level', 'pod_id', 'pod_name', 'clusterId', 'clusterName', '__topic__'], dtype='object')```
```